In [203]:
import json

prev_catalog = json.load(open("old-catalog.json"))
prev_catalog_shrtname_2_id = {item["content_name"]: item["content_id"] for item in prev_catalog}

In [204]:
import os
import requests

api_token = open('api_token.txt').read().strip()
for filename in os.listdir("items"):
    if not filename.endswith(".json"):
        continue
    item = json.load(open(os.path.join("items", filename)))
    if "paws_id" in item:
        continue
    
    if item['identity']['id'] in prev_catalog_shrtname_2_id:
        item['paws_id'] = str(prev_catalog_shrtname_2_id[item['identity']['id']])
        json.dump(item, open(os.path.join("items", filename), "w"), indent=2)
        print("./items/" + filename, "updated with paws_id", item['paws_id'])
    elif item['identity']['title'] in prev_catalog_shrtname_2_id:
        item['paws_id'] = str(prev_catalog_shrtname_2_id[item['identity']['title']])
        json.dump(item, open(os.path.join("items", filename), "w"), indent=2)
        print("./items/" + filename, "updated with paws_id", item['paws_id'])
    else:
        # -- DUPLICATE titles
        # if item['id'] in ["699249d87e3947753a7663d8","699249d87e3947753a7663de","699249d87e3947753a7663d2","699249d97e3947753a7663e2"]:
        #     item['identity']['title'] += f" [{item['identity']['type']}]"
        print("./items/" + filename, "not found in old catalog, syncing ...")
        response = requests.put(
            f"http://adapt2.sis.pitt.edu/next.course-authoring/api/slc-items-api/{item['id']}/sync-to-aggregate",
            headers={'api-token': api_token}
        )
        print("sync response:", response.status_code, response.text)
        if response.status_code != 200:
            print("Failed to sync item", item['id'], "skipping...")
            continue
        item['paws_id'] = response.json()['paws_id']
        json.dump(item, open(os.path.join("items", filename), "w"), indent=2)
        print("./items/" + filename, "updated with paws_id", item['paws_id'])

print("done updating items with paws_id!")

done updating items with paws_id!
